# 📊 Konversi Excel → JSON Dashboard
**KANWIL III SUMBAGSEL — Laba Usaha per Outlet**

---
### Cara pakai:
1. Upload file Excel Anda di **Sel 2** (klik tombol upload)
2. Isi nama bulan dan nama file di **Sel 3**
3. Klik **Runtime → Run all**
4. Download hasil JSON dari panel Files (📁) di kiri
5. Upload JSON ke folder `data/` di GitHub

In [ ]:
# ── SEL 1: Install library ───────────────────────────────────
# Jalankan sekali. Colab sudah punya pandas, tapi openpyxl perlu dicek.
!pip install -q openpyxl
print('✅ Library siap')

In [ ]:
# ── SEL 2: Upload file Excel ─────────────────────────────────
from google.colab import files

print('📂 Silakan pilih semua file Excel yang ingin dikonversi...')
uploaded = files.upload()

print(f'\n✅ {len(uploaded)} file berhasil diupload:')
for nama in uploaded:
    print(f'   • {nama}')

In [ ]:
# ── SEL 3: KONFIGURASI — edit bagian ini ─────────────────────
#
# Isi sesuai file yang sudah diupload di Sel 2
# Format: 'nama_bulan' : 'nama_file.xlsx'
#
DAFTAR_BULAN = {
    'januari'  : 'LABA_Januari_2026.xlsx',
    'februari' : 'LABA_Februari_2026.xlsx',
    'maret'    : 'LABA_Maret_2026.xlsx',
    'april'    : 'LABA_April_2026.xlsx',
    'mei'      : 'LABA_Usaha_Bopo_ROA_ROE_Channeling_Per_Outlet_sd_31_MEI_2026_-_Copy.xlsx',
}

# Hapus bulan yang belum ada filenya
DAFTAR_BULAN = {b: f for b, f in DAFTAR_BULAN.items() if f in uploaded}

print('📋 Bulan yang akan diproses:')
for b, f in DAFTAR_BULAN.items():
    print(f'   • {b.capitalize()} → {f}')

if not DAFTAR_BULAN:
    print('⚠️  Tidak ada file yang cocok. Periksa nama file di atas.')

In [ ]:
# ── SEL 4: Konfigurasi baris Excel ───────────────────────────
# Edit bagian ini HANYA jika format/posisi baris Excel berubah.
# Jika format sama setiap bulan, tidak perlu diubah.

SHEET_NAME = 'LABA BOPO PER OUTLET '   # perhatikan spasi di akhir

# Mapping kolom (0-based)
COL = dict(
    kode=0, nama=1,
    pendapatan=2, biaya=3, bopo=4, laba=5,
    aset=6, ekuitas=7,
    target_laba_mei=9,
    target_pend_mei=11,
    target_bopo=14,
    ach_bopo=15,
    ach_laba_mei=17,
    ach_pend_mei=19,
    roa=22, roe=23,
)

# Baris TOTAL per area (0-based)
AREA_TOTAL_ROW = {
    'AREA LAMPUNG'   : 115,
    'AREA PALEMBANG' : 337,
    'AREA JAMBI'     : 508,
}

# Rentang baris data [start, end)
AREA_DATA_RANGE = {
    'AREA LAMPUNG'   : (21,  115),
    'AREA PALEMBANG' : (169, 337),
    'AREA JAMBI'     : (362, 508),
}

# Baris TOTAL per CP
CP_TOTAL_ROWS = {
    'AREA LAMPUNG'   : [33, 41, 51, 63, 77, 86, 102, 114],
    'AREA PALEMBANG' : [182, 190, 202, 213, 225, 237, 250, 258, 274, 287, 298, 310, 318, 335],
    'AREA JAMBI'     : [381, 397, 411, 425, 441, 454, 467, 481, 493, 507],
}

GRAND_TOTAL_ROW = 509

print('✅ Konfigurasi baris siap')

In [ ]:
# ── SEL 5: Fungsi konversi ────────────────────────────────────
import pandas as pd
import json
import os
import warnings
warnings.filterwarnings('ignore')

def safe_float(val):
    if val is None: return 0.0
    try:
        f = float(val)
        return 0.0 if f != f else f
    except: return 0.0

def row_name(df, row):
    raw = df.iloc[row, COL['nama']]
    if raw is None or (isinstance(raw, float) and raw != raw): return ''
    name = str(raw).strip()
    return name.split(':', 1)[1].strip() if ':' in name else name

def extract_row(df, row):
    return {k: safe_float(df.iloc[row, v]) for k, v in COL.items() if k not in ('kode','nama')}

def convert(excel_path):
    df = pd.read_excel(excel_path, sheet_name=SHEET_NAME, header=None)
    print(f'   → {df.shape[0]} baris, {df.shape[1]} kolom')

    result = {
        'grand_total': {'name': 'GRAND TOTAL KANWIL PALEMBANG', **extract_row(df, GRAND_TOTAL_ROW)},
        'areas': {}
    }

    for area_name, total_row in AREA_TOTAL_ROW.items():
        cp_rows = CP_TOTAL_ROWS[area_name]
        start, end = AREA_DATA_RANGE[area_name]
        outlets_by_cp = {r: [] for r in cp_rows}

        for idx in range(start, end):
            kode = str(df.iloc[idx, COL['kode']]).strip() if df.iloc[idx, COL['kode']] is not None else ''
            try: int(float(kode))
            except: continue
            nama = row_name(df, idx)
            if not nama: continue
            next_cp = next((r for r in cp_rows if r > idx), None)
            if next_cp:
                outlets_by_cp[next_cp].append({'code': kode, 'name': nama, **extract_row(df, idx)})

        cp_list = []
        for cp_row in cp_rows:
            cp_name = row_name(df, cp_row)
            if cp_name:
                cp_list.append({'name': cp_name, 'outlets': outlets_by_cp.get(cp_row, []), **extract_row(df, cp_row)})

        result['areas'][area_name] = {'name': area_name, 'cp_list': cp_list, **extract_row(df, total_row)}
        total_outlets = sum(len(cp['outlets']) for cp in cp_list)
        print(f'   ✓ {area_name}: {len(cp_list)} CP, {total_outlets} outlet')

    return result

print('✅ Fungsi konversi siap')

In [ ]:
# ── SEL 6: Proses semua bulan & download hasilnya ─────────────
os.makedirs('data', exist_ok=True)
hasil = []

for bulan, nama_file in DAFTAR_BULAN.items():
    print(f'\n🔄 Memproses {bulan.upper()} — {nama_file}')
    try:
        data = convert(nama_file)
        out_path = f'data/{bulan}.json'
        with open(out_path, 'w', encoding='utf-8') as f:
            json.dump(data, f, ensure_ascii=False, separators=(',', ':'))
        size_kb = os.path.getsize(out_path) / 1024
        print(f'   💾 Tersimpan: {out_path} ({size_kb:.1f} KB)')
        hasil.append((bulan, out_path, size_kb, '✅'))
    except Exception as e:
        print(f'   ❌ Gagal: {e}')
        hasil.append((bulan, '-', 0, f'❌ {e}'))

# Ringkasan
print('\n' + '='*50)
print('📋 RINGKASAN HASIL')
print('='*50)
for bulan, path, size, status in hasil:
    print(f'{status}  {bulan.capitalize():<12} → {path:<20} {size:.1f} KB' if size else f'{status}  {bulan.capitalize()}')

print('\n⬇️  Download file JSON dari panel Files (ikon 📁 di sidebar kiri)')
print('   Lalu upload ke folder data/ di repo GitHub Anda')

In [ ]:
# ── SEL 7 (OPSIONAL): Download semua JSON sekaligus sebagai ZIP ──
import zipfile
from google.colab import files

zip_path = 'data_dashboard.zip'
with zipfile.ZipFile(zip_path, 'w') as zf:
    for bulan, path, size, status in hasil:
        if status == '✅':
            zf.write(path)

print(f'📦 ZIP siap: {zip_path}')
print('   Isi ZIP:')
with zipfile.ZipFile(zip_path) as zf:
    for name in zf.namelist():
        print(f'   • {name}')

files.download(zip_path)
print('\n✅ Download dimulai!')

---
## Langkah selanjutnya setelah download JSON

1. **Extract ZIP** → ambil file-file `.json` di dalam folder `data/`
2. **Upload ke GitHub** → masuk ke repo → folder `data/` → drag & drop file JSON
3. **Edit `index.html`** → tambah opsi bulan yang belum ada:
   ```html
   <option value="januari">Januari 2026</option>
   <option value="februari">Februari 2026</option>
   ```
4. **Commit** → dashboard otomatis update ✅

---
**Bulan berikutnya:** Upload Excel baru → isi `DAFTAR_BULAN` di Sel 3 → Run all → download JSON → push ke GitHub.